In [ ]:
# In[1]:

import pandas as pd

# Load CSVs (they are in current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (keep original integer column too)
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['ts'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Ensure numeric 'value' columns where applicable
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) metric.csv: aggregate per (cmdb_id, kpi_name) -> count, mean, p95
metric_agg = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='count', mean='mean')
)

# Compute global P95 per group (using groupby + quantile)
p95_series = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('p95').reset_index()
metric_agg = metric_agg.merge(p95_series, on=['cmdb_id', 'kpi_name'])

# Sort by count desc and take top 20
metric_agg_top20 = metric_agg.sort_values('count', ascending=False).head(20)
# Round numeric columns for compactness
metric_agg_top20['mean'] = metric_agg_top20['mean'].round(6)
metric_agg_top20['p95'] = metric_agg_top20['p95'].round(6)

# Distinct cmdb_id and kpi_name lists (short lists)
metric_cmdbs = metric_df['cmdb_id'].dropna().unique().tolist()
metric_kpis = metric_df['kpi_name'].dropna().unique().tolist()

# 2) trace.csv: distinct trace_name values and distinct cmdb_id values and counts per pair (top20)
trace_trace_names = trace_df['trace_name'].dropna().unique().tolist()
trace_cmdbs = trace_df['cmdb_id'].dropna().unique().tolist()

trace_pairs = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(count=('value', 'count'))
    .sort_values('count', ascending=False)
)
trace_pairs_top20 = trace_pairs.head(20)

# Top 10 rows of trace.csv as sample
trace_df_head10 = trace_df.head(10)

# 3) log.csv: distinct log_name values and distinct cmdb_id values and counts per pair (top20)
log_log_names = log_df['log_name'].dropna().unique().tolist()
log_cmdbs = log_df['cmdb_id'].dropna().unique().tolist()

log_pairs = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(count=('value', 'count'))
    .sort_values('count', ascending=False)
)
log_pairs_top20 = log_pairs.head(20)

# Top 10 rows of log.csv as sample
log_df_head10 = log_df.head(10)

# 4) error_logs.csv: top 10 rows, distinct cmdb_id values present, total number of error log records
error_df_head10 = error_df.head(10)
error_cmdbs = error_df['cmdb_id'].dropna().unique().tolist()
error_count = len(error_df)

# Present compact outputs as requested (variables shown)
metric_agg_top20, metric_cmdbs[:50], metric_kpis[:50], \
trace_trace_names, trace_cmdbs, trace_pairs_top20, trace_df_head10, \
log_log_names, log_cmdbs, log_pairs_top20, log_df_head10, \
error_df_head10, error_cmdbs, error_count

```
Out[1]:
```
summary = (
    "Summary of telemetry files (compact):\n\n"
    "1) metric.csv\n"
    "- Top (cmdb_id, kpi_name) pairs by sample count are dominated by: carts, carts-db, catalogue, catalogue-db (top pairs all have count=25).\n"
    "- Distinct cmdb_id values (sample): ['carts', 'carts-db', 'catalogue', 'catalogue-db', 'front-end', 'orders', 'orders-db', 'payment', 'queue-master', 'rabbitmq', 'rabbitmq-exporter', 'session-db', 'shipping', 'user', 'user-db']\n"
    "- Distinct kpi_name values (sample): ['cpu', 'diskio', 'latency-50', 'latency-90', 'mem', 'socket', 'workload', 'error']\n\n"
    "2) trace.csv\n"
    "- No trace telemetry records found (trace.csv is empty).\n\n"
    "3) log.csv\n"
    "- Distinct log_name values: ['log.error_count', 'log.row_count']\n"
    "- Distinct cmdb_id values (sample): ['carts', 'carts-db', 'catalogue', 'front-end', 'orders', 'payment', 'queue-master', 'shipping', 'user', 'session-db']\n"
    "- Counts per (cmdb_id, log_name): many services (front-end, catalogue, queue-master, shipping, orders, payment, user) have 25 samples; session-db has 5, carts has 19, carts-db has 24.\n"
    "- Top-10 sample rows include examples such as front-end log.row_count=1828, catalogue log.row_count=153, orders log.row_count=112.\n\n"
    "4) error_logs.csv\n"
    "- No error log records found (0 rows).\n\n"
    "Overall conclusion: metric and log telemetry are available for many services (notably carts, carts-db, catalogue, front-end, orders, payment, etc.). Trace and error_log data are absent in this dataset."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count          mean           p95
0          carts         cpu     25  1.202554e+00  1.322364e+00
1          carts      diskio     25  0.000000e+00  0.000000e+00
2          carts  latency-50     25  1.517200e-02  1.676700e-02
3          carts  latency-90     25  2.651100e-02  3.661800e-02
4          carts         mem     25  2.074931e+08  2.074132e+08
5          carts      socket     25  1.379983e+01  1.463667e+01
6          carts    workload     25  8.183917e+00  8.562903e+00
7       carts-db         cpu     25  3.817269e+00  4.259195e+00
8       carts-db      diskio     25  2.018205e+06  2.266218e+06
9       carts-db         mem     25  8.517559e+07  8.769277e+07
10      carts-db      socket     25  1.000000e+01  1.000000e+01
11     catalogue         cpu     25  1.628250e-01  1.725040e-01
12     catalogue  latency-50     25  3.022000e-03  3.057000e-03
13     catalogue  latency-90     25  4.639000e-03  4.702000e-03
14     catalogue         mem     25  6.111119e+06  6.221469e+06
15     catalogue      socket     25  6.000000e+00  6.000000e+00
16     catalogue    workload     25  4.087262e+00  4.225960e+00
17  catalogue-db         cpu     25  1.925160e-01  2.041400e-01
18  catalogue-db      diskio     25  0.000000e+00  0.000000e+00
19  catalogue-db         mem     25  4.306940e+08  4.307110e+08, ['carts', 'carts-db', 'catalogue', 'catalogue-db', 'front-end', 'orders', 'orders-db', 'payment', 'queue-master', 'rabbitmq', 'rabbitmq-exporter', 'session-db', 'shipping', 'user', 'user-db'], ['cpu', 'diskio', 'latency-50', 'latency-90', 'mem', 'socket', 'workload', 'error'], [], [], Empty DataFrame
Columns: [cmdb_id, trace_name, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value, ts]
Index: [], ['log.error_count', 'log.row_count'], ['carts', 'carts-db', 'catalogue', 'front-end', 'orders', 'payment', 'queue-master', 'shipping', 'user', 'session-db'],          cmdb_id         log_name  count
7      front-end    log.row_count     25
6      front-end  log.error_count     25
5      catalogue    log.row_count     25
4      catalogue  log.error_count     25
12  queue-master  log.error_count     25
13  queue-master    log.row_count     25
17      shipping    log.row_count     25
16      shipping  log.error_count     25
8         orders  log.error_count     25
9         orders    log.row_count     25
10       payment  log.error_count     25
11       payment    log.row_count     25
18          user  log.error_count     25
19          user    log.row_count     25
2       carts-db  log.error_count     24
3       carts-db    log.row_count     24
0          carts  log.error_count     19
1          carts    log.row_count     19
15    session-db    log.row_count      5
14    session-db  log.error_count      5,     timestamp    cmdb_id         log_name  value                        ts
0  1705847760      carts  log.error_count      0 2024-01-21 14:36:00+00:00
1  1705847760      carts    log.row_count     50 2024-01-21 14:36:00+00:00
2  1705847760   carts-db  log.error_count      0 2024-01-21 14:36:00+00:00
3  1705847760   carts-db    log.row_count      1 2024-01-21 14:36:00+00:00
4  1705847760  catalogue  log.error_count      0 2024-01-21 14:36:00+00:00
5  1705847760  catalogue    log.row_count    153 2024-01-21 14:36:00+00:00
6  1705847760  front-end  log.error_count      0 2024-01-21 14:36:00+00:00
7  1705847760  front-end    log.row_count   1828 2024-01-21 14:36:00+00:00
8  1705847760     orders  log.error_count      0 2024-01-21 14:36:00+00:00
9  1705847760     orders    log.row_count    112 2024-01-21 14:36:00+00:00, Empty DataFrame
Columns: [timestamp, cmdb_id, message, error, ts]
Index: [], [], 0)```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse metric_df from kernel
_metric = metric_df
_metric['value'] = pd.to_numeric(_metric['value'], errors='coerce')
if 'ts' not in _metric.columns:
    _metric['ts'] = pd.to_datetime(_metric['timestamp'], unit='s', utc=True)

# 1) Global P95 per (cmdb_id, kpi_name)
p95_df = (
    _metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95'})
)
p95_table_top50 = p95_df.sort_values(['cmdb_id', 'kpi_name']).head(50)
p95_table_top50['p95'] = p95_table_top50['p95'].round(6)

# 2) Filter to incident window and detect anomalies using global P95
start = pd.to_datetime("2024-01-21 14:33:03", utc=True)
end   = pd.to_datetime("2024-01-21 15:03:03", utc=True)
window_df = _metric[(_metric['ts'] >= start) & (_metric['ts'] <= end)].copy()

kpis_of_interest = ['cpu', 'mem', 'diskio', 'latency-50', 'latency-90', 'socket', 'workload', 'error']

# Merge global p95 into windowed data
window_with_p95 = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')
window_with_p95 = window_with_p95[window_with_p95['kpi_name'].isin(kpis_of_interest)].copy()

# Mark anomalies where p95 exists
window_with_p95['is_anomaly'] = False
mask_has_p95 = window_with_p95['p95'].notna()
window_with_p95.loc[mask_has_p95, 'is_anomaly'] = window_with_p95.loc[mask_has_p95, 'value'] >= window_with_p95.loc[mask_has_p95, 'p95']

# Aggregations per pair
agg_basic = (
    window_with_p95
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        anomaly_count = ('is_anomaly', 'sum'),
        max_value_in_window = ('value', 'max'),
        p95 = ('p95', 'first')
    )
)

# Earliest anomaly timestamp per pair (from rows where is_anomaly==True)
anom_times = (
    window_with_p95[window_with_p95['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(earliest_anomaly_ts = ('ts', 'min'))
)
# Merge earliest times (may be missing)
agg_df = agg_basic.merge(anom_times, on=['cmdb_id', 'kpi_name'], how='left')

# severity_ratio = max_value_in_window / p95 (safe)
agg_df['severity_ratio'] = np.where(agg_df['p95'] > 0, agg_df['max_value_in_window'] / agg_df['p95'], np.nan)

# Rounding for compactness
agg_df['p95'] = agg_df['p95'].round(6)
agg_df['max_value_in_window'] = agg_df['max_value_in_window'].round(6)
agg_df['severity_ratio'] = agg_df['severity_ratio'].round(6)

# Sort as requested and take top20
anomalies_agg_top20 = agg_df.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).head(20)

# 3) If no anomalies found, repeat using P90
total_anomalies = int(agg_df['anomaly_count'].sum())
used_threshold = 'p95'
if total_anomalies == 0:
    p90_df = (
        _metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.90)
        .rename(columns={'value': 'p90'})
    )
    window_with_p90 = window_df.merge(p90_df, on=['cmdb_id', 'kpi_name'], how='left')
    window_with_p90 = window_with_p90[window_with_p90['kpi_name'].isin(kpis_of_interest)].copy()
    window_with_p90['is_anomaly'] = False
    mask_has_p90 = window_with_p90['p90'].notna()
    window_with_p90.loc[mask_has_p90, 'is_anomaly'] = window_with_p90.loc[mask_has_p90, 'value'] >= window_with_p90.loc[mask_has_p90, 'p90']
    agg_basic_p90 = (
        window_with_p90
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_count = ('is_anomaly', 'sum'),
            max_value_in_window = ('value', 'max'),
            p90 = ('p90', 'first')
        )
    )
    anom_times_p90 = (
        window_with_p90[window_with_p90['is_anomaly']]
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(earliest_anomaly_ts = ('ts', 'min'))
    )
    agg_df_p90 = agg_basic_p90.merge(anom_times_p90, on=['cmdb_id', 'kpi_name'], how='left')
    agg_df_p90['severity_ratio'] = np.where(agg_df_p90['p90'] > 0, agg_df_p90['max_value_in_window'] / agg_df_p90['p90'], np.nan)
    agg_df_p90['p90'] = agg_df_p90['p90'].round(6)
    agg_df_p90['max_value_in_window'] = agg_df_p90['max_value_in_window'].round(6)
    agg_df_p90['severity_ratio'] = agg_df_p90['severity_ratio'].round(6)
    anomalies_agg_top20 = agg_df_p90.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).head(20)
    used_threshold = 'p90'

# Prepare top 3 pairs and sample up to first 10 metric rows in window for each
top3 = anomalies_agg_top20.head(3)[['cmdb_id', 'kpi_name']].copy()
sample_rows_list = []
for _, row in top3.iterrows():
    cmdb = row['cmdb_id']; kpi = row['kpi_name']
    df_pair = window_df[(window_df['cmdb_id']==cmdb) & (window_df['kpi_name']==kpi)].sort_values('ts').head(10)
    if not df_pair.empty:
        sample = df_pair[['timestamp','cmdb_id','kpi_name','value','ts']].copy()
        sample['pair'] = f"{cmdb}||{kpi}"
        sample_rows_list.append(sample)
if sample_rows_list:
    top3_samples = pd.concat(sample_rows_list, ignore_index=True)
else:
    top3_samples = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','ts','pair'])

# Final compact outputs (variables to display)
p95_table_top50, used_threshold, anomalies_agg_top20, top3_samples

```
Out[2]:
```
Summary of results (plain English, compact):

- Global P95 (computed over the full metric series): produced per (cmdb_id, kpi_name). Examples from the top-50: 
  - carts.cpu p95 ≈ 1.322364
  - carts.mem p95 ≈ 2.074132e+08
  - carts.socket p95 ≈ 14.63667
  - queue-master.mem p95 ≈ 2.821424e+08
  (full per-pair P95 table was computed and stored.)

- Incident-window anomaly detection used global P95 as threshold (window = 2024-01-21 14:33:03 → 2024-01-21 15:03:03 UTC).

- Main findings (top pairs by anomaly_count then severity):
  - Many services show the socket KPI exactly at or equal to their P95 for the whole inspected window (anomaly_count = 25, severity_ratio = 1.0). Examples: carts-db.socket, catalogue.socket, catalogue-db.socket, orders-db.socket, payment.socket, queue-master.socket, rabbitmq.socket, user.socket, user-db.socket, session-db.socket, etc. Earliest breach for those = 2024-01-21 14:36:00+00:00.
  - Some KPIs have p95 == 0 (diskio, error for some services); those produce NaN severity_ratio and are not informative for >P95 detection.
  - Notable spikes exceeding P95 (severity_ratio > 1):
    - shipping.cpu: anomaly_count = 2, max_in_window ≈ 2.032583, p95 ≈ 1.119742, earliest breach 2024-01-21 14:42:00+00:00, severity_ratio ≈ 1.8152.
    - session-db.diskio: anomaly_count = 2, max_in_window ≈ 27570.99, p95 ≈ 17812.06, earliest breach 2024-01-21 14:39:00+00:00, severity_ratio ≈ 1.5479.

- Top 3 (cmdb_id, kpi_name) from the ranked anomalies (examples) and notes on rows:
  1. carts-db — socket: anomaly_count = 25, max = 10.0, p95 = 10.0, earliest = 2024-01-21 14:36:00+00:00. Sample rows show socket=10.0 repeated at consecutive minute timestamps in the window.
  2. catalogue — socket: anomaly_count = 25, max = 6.0, p95 = 6.0, earliest = 2024-01-21 14:36:00+00:00. Sample rows show repeated values at consecutive minutes.
  3. catalogue-db — socket: anomaly_count = 25, max = 4.0, p95 = 4.0, earliest = 2024-01-21 14:36:00+00:00. Sample rows likewise are consecutive.

- Interpretation / concise conclusions:
  - Many services have their socket metric at the P95 level consistently across the incident window — this looks like a system-wide socket-level saturation (values hit their historical 95th percentile repeatedly).
  - Two clear spikes above P95 were observed for shipping.cpu and session-db.diskio (severity_ratio >1.5), which are localized anomalies worth inspecting further.
  - Several diskio/error KPIs have p95 == 0 in the dataset; their thresholds are not useful without additional context.

If you want, next steps I can run:
- Expand samples for specific top pairs to see continuity, or
- Drill into shipping.cpu and session-db.diskio time series to inspect timestamps and contextual metrics.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name           p95
0          carts         cpu  1.322364e+00
1          carts      diskio  0.000000e+00
2          carts  latency-50  1.676700e-02
3          carts  latency-90  3.661800e-02
4          carts         mem  2.074132e+08
..           ...         ...           ...
45       payment    workload  2.126780e+00
46  queue-master         cpu  6.685540e-01
47  queue-master      diskio  6.777485e+04
48  queue-master         mem  2.821424e+08
49  queue-master      socket  3.000000e+00

[50 rows x 3 columns], 'p95',               cmdb_id kpi_name  anomaly_count  max_value_in_window           p95       earliest_anomaly_ts  severity_ratio
10           carts-db   socket             25         1.000000e+01  1.000000e+01 2024-01-21 14:36:00+00:00        1.000000
15          catalogue   socket             25         6.000000e+00  6.000000e+00 2024-01-21 14:36:00+00:00        1.000000
20       catalogue-db   socket             25         4.000000e+00  4.000000e+00 2024-01-21 14:36:00+00:00        1.000000
39          orders-db   socket             25         5.000000e+00  5.000000e+00 2024-01-21 14:36:00+00:00        1.000000
44            payment   socket             25         5.000000e+00  5.000000e+00 2024-01-21 14:36:00+00:00        1.000000
49       queue-master   socket             25         3.000000e+00  3.000000e+00 2024-01-21 14:36:00+00:00        1.000000
53           rabbitmq   socket             25         1.100000e+01  1.100000e+01 2024-01-21 14:36:00+00:00        1.000000
55  rabbitmq-exporter      mem             25         4.816896e+06  4.816896e+06 2024-01-21 14:36:00+00:00        1.000000
56  rabbitmq-exporter   socket             25         1.000000e+00  1.000000e+00 2024-01-21 14:36:00+00:00        1.000000
60         session-db   socket             25         3.000000e+00  3.000000e+00 2024-01-21 14:36:00+00:00        1.000000
72               user   socket             25         1.800000e+01  1.800000e+01 2024-01-21 14:36:00+00:00        1.000000
77            user-db   socket             25         8.000000e+00  8.000000e+00 2024-01-21 14:36:00+00:00        1.000000
1               carts   diskio             25         0.000000e+00  0.000000e+00 2024-01-21 14:36:00+00:00             NaN
18       catalogue-db   diskio             25         0.000000e+00  0.000000e+00 2024-01-21 14:36:00+00:00             NaN
22          front-end    error             25         0.000000e+00  0.000000e+00 2024-01-21 14:36:00+00:00             NaN
30             orders    error             25         0.000000e+00  0.000000e+00 2024-01-21 14:36:00+00:00             NaN
51           rabbitmq   diskio             25         0.000000e+00  0.000000e+00 2024-01-21 14:36:00+00:00             NaN
68               user    error             25         0.000000e+00  0.000000e+00 2024-01-21 14:36:00+00:00             NaN
61           shipping      cpu              2         2.032583e+00  1.119742e+00 2024-01-21 14:42:00+00:00        1.815224
58         session-db   diskio              2         2.757099e+04  1.781206e+04 2024-01-21 14:39:00+00:00        1.547883,      timestamp       cmdb_id kpi_name  value                        ts                  pair
0   1705847760      carts-db   socket   10.0 2024-01-21 14:36:00+00:00      carts-db||socket
1   1705847820      carts-db   socket   10.0 2024-01-21 14:37:00+00:00      carts-db||socket
2   1705847880      carts-db   socket   10.0 2024-01-21 14:38:00+00:00      carts-db||socket
3   1705847940      carts-db   socket   10.0 2024-01-21 14:39:00+00:00      carts-db||socket
4   1705848000      carts-db   socket   10.0 2024-01-21 14:40:00+00:00      carts-db||socket
..         ...           ...      ...    ...                       ...                   ...
25  1705848060  catalogue-db   socket    4.0 2024-01-21 14:41:00+00:00  catalogue-db||socket
26  1705848120  catalogue-db   socket    4.0 2024-01-21 14:42:00+00:00  catalogue-db||socket
27  1705848180  catalogue-db   socket    4.0 2024-01-21 14:43:00+00:00  catalogue-db||socket
28  1705848240  catalogue-db   socket    4.0 2024-01-21 14:44:00+00:00  catalogue-db||socket
29  1705848300  catalogue-db   socket    4.0 2024-01-21 14:45:00+00:00  catalogue-db||socket

[30 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse cached dataframes: metric_df, log_df, error_df, trace_df, p95_df assumed present
# Ensure numeric and ts columns exist
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
for df in (metric_df, log_df):
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Incident window
start = pd.to_datetime("2024-01-21 14:33:03", utc=True)
end   = pd.to_datetime("2024-01-21 15:03:03", utc=True)

# 1) Metric rows in window for shipping.cpu and session-db.diskio (up to 50 rows each)
shipping_cpu_rows = (
    metric_df[
        (metric_df['cmdb_id'] == 'shipping') &
        (metric_df['kpi_name'] == 'cpu') &
        (metric_df['ts'] >= start) &
        (metric_df['ts'] <= end)
    ]
    .sort_values('ts')
    .head(50)[['timestamp','cmdb_id','kpi_name','value','ts']]
)

sessiondb_diskio_rows = (
    metric_df[
        (metric_df['cmdb_id'] == 'session-db') &
        (metric_df['kpi_name'] == 'diskio') &
        (metric_df['ts'] >= start) &
        (metric_df['ts'] <= end)
    ]
    .sort_values('ts')
    .head(50)[['timestamp','cmdb_id','kpi_name','value','ts']]
)

# 2) Aggregate log.error_count within the incident window
window_log = log_df[(log_df['ts'] >= start) & (log_df['ts'] <= end)].copy()
error_logs_focus = window_log[window_log['log_name'] == 'log.error_count'].copy()

# total_error_count (sum), max_error (max), earliest_error_ts where value>0
agg_error_sum_max = (
    error_logs_focus
    .groupby('cmdb_id', as_index=False)
    .agg(total_error_count=('value','sum'), max_error=('value','max'))
)

earliest_positive = (
    error_logs_focus[error_logs_focus['value'] > 0]
    .groupby('cmdb_id', as_index=False)
    .agg(earliest_error_ts=('ts','min'))
)

error_agg_df = agg_error_sum_max.merge(earliest_positive, on='cmdb_id', how='left')
# Keep only cmdb_id with total_error_count > 0, sort desc
error_agg_df = error_agg_df[error_agg_df['total_error_count'] > 0].sort_values('total_error_count', ascending=False).head(20)

# Short list of cmdb_id values that have any log.error_count>0
error_cmdbs_list = error_agg_df['cmdb_id'].dropna().unique().tolist()

# 3) Socket KPIs within incident window: anomaly_count (value >= global p95), earliest_anomaly_ts, max_value_in_window
# Ensure global p95 table exists; if not, compute it (rule 9)
if 'p95_df' not in globals():
    p95_df = metric_df.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'p95'})

# Filter socket rows in window and join global p95
socket_window = (
    metric_df[
        (metric_df['kpi_name'] == 'socket') &
        (metric_df['ts'] >= start) &
        (metric_df['ts'] <= end)
    ]
    .merge(p95_df[p95_df['kpi_name']=='socket'], on=['cmdb_id','kpi_name'], how='left')
    .copy()
)

# Determine anomalies where p95 exists
socket_window['is_anomaly'] = False
mask_p95 = socket_window['p95'].notna()
socket_window.loc[mask_p95, 'is_anomaly'] = socket_window.loc[mask_p95, 'value'] >= socket_window.loc[mask_p95, 'p95']

# Aggregations per (cmdb_id, kpi_name)
socket_agg = (
    socket_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        anomaly_count = ('is_anomaly', 'sum'),
        earliest_anomaly_ts = ('ts', lambda s: s[socket_window.loc[s.index, 'is_anomaly']].min() if len(s)>0 else pd.NaT),
        max_value_in_window = ('value','max')
    )
)

# Clean earliest_anomaly_ts by recomputing properly using rows where is_anomaly True
earliest = (
    socket_window[socket_window['is_anomaly']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(earliest_anomaly_ts=('ts','min'))
)
socket_agg = socket_agg.drop(columns=['earliest_anomaly_ts']).merge(earliest, on=['cmdb_id','kpi_name'], how='left')

# Keep top 50 rows sorted by anomaly_count desc then max_value_in_window desc
socket_agg_top50 = socket_agg.sort_values(['anomaly_count','max_value_in_window'], ascending=[False, False]).head(50)

# Distinct earliest_anomaly_ts values seen for socket KPIs
socket_earliest_ts_set = socket_agg_top50['earliest_anomaly_ts'].dropna().sort_values().unique().tolist()

# Final compact outputs
shipping_cpu_rows, sessiondb_diskio_rows, error_agg_df, error_cmdbs_list, socket_agg_top50[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','max_value_in_window']].head(50), socket_earliest_ts_set

```
Out[3]:
```
summary = (
    "Compact findings (incident window 2024-01-21 14:33:03 → 2024-01-21 15:03:03 UTC):\n\n"
    "1) shipping.cpu (metric rows in window): 25 samples, values ≈ 0.229 → 0.611 (no large spike visible in these rows).\n\n"
    "2) session-db.diskio (metric rows in window): 25 samples, mostly 0.0 but with clear spikes at:\n"
    "   - 2024-01-21 14:39:00 UTC: 18173.33\n"
    "   - 2024-01-21 14:59:00 UTC: 12884.89\n"
    "   - 2024-01-21 15:00:00 UTC: 27570.99\n"
    "   These are pronounced disk I/O anomalies in the session-db series.\n\n"
    "3) log.error_count in the window: no cmdb_id had total_error_count > 0 (no error_count activity found).\n\n"
    "4) socket KPI summary (within window): many services hit or exceeded their global P95 at the same time:\n"
    "   - A large group (e.g., user, rabbitmq, carts-db, user-db, catalogue, orders-db, payment, catalogue-db, queue-master, session-db, rabbitmq-exporter, etc.) show anomaly_count=25 with earliest breach at 2024-01-21 14:36:00 UTC — indicating simultaneous socket-level saturation.\n"
    "   - A few services (carts, front-end, orders, shipping) show shorter anomalies (anomaly_count=2) with earliest breaches at 14:37, 14:49, 14:53, 14:55 UTC respectively.\n\n"
    "Overall interpretation: the dataset shows widespread socket metric saturation beginning ~14:36 UTC and a clear disk I/O spike on session-db (multiple large values). No error log counts were recorded in the window. These are the highest-priority signals to investigate further (session-db disk I/O and the system-wide socket behavior)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       timestamp   cmdb_id kpi_name     value                        ts
61    1705847760  shipping      cpu  0.345377 2024-01-21 14:36:00+00:00
139   1705847820  shipping      cpu  0.299475 2024-01-21 14:37:00+00:00
217   1705847880  shipping      cpu  0.409781 2024-01-21 14:38:00+00:00
295   1705847940  shipping      cpu  0.606087 2024-01-21 14:39:00+00:00
373   1705848000  shipping      cpu  0.326656 2024-01-21 14:40:00+00:00
...          ...       ...      ...       ...                       ...
1621  1705848960  shipping      cpu  0.325663 2024-01-21 14:56:00+00:00
1699  1705849020  shipping      cpu  0.310698 2024-01-21 14:57:00+00:00
1777  1705849080  shipping      cpu  0.310869 2024-01-21 14:58:00+00:00
1855  1705849140  shipping      cpu  0.610788 2024-01-21 14:59:00+00:00
1933  1705849200  shipping      cpu  0.228887 2024-01-21 15:00:00+00:00

[25 rows x 5 columns],        timestamp     cmdb_id kpi_name         value                        ts
58    1705847760  session-db   diskio      0.000000 2024-01-21 14:36:00+00:00
136   1705847820  session-db   diskio      0.000000 2024-01-21 14:37:00+00:00
214   1705847880  session-db   diskio      0.000000 2024-01-21 14:38:00+00:00
292   1705847940  session-db   diskio  18173.332627 2024-01-21 14:39:00+00:00
370   1705848000  session-db   diskio      0.000000 2024-01-21 14:40:00+00:00
...          ...         ...      ...           ...                       ...
1618  1705848960  session-db   diskio      0.000000 2024-01-21 14:56:00+00:00
1696  1705849020  session-db   diskio      0.000000 2024-01-21 14:57:00+00:00
1774  1705849080  session-db   diskio      0.000000 2024-01-21 14:58:00+00:00
1852  1705849140  session-db   diskio  12884.892346 2024-01-21 14:59:00+00:00
1930  1705849200  session-db   diskio  27570.987422 2024-01-21 15:00:00+00:00

[25 rows x 5 columns], Empty DataFrame
Columns: [cmdb_id, total_error_count, max_error, earliest_error_ts]
Index: [], [],               cmdb_id kpi_name  anomaly_count       earliest_anomaly_ts  max_value_in_window
13               user   socket             25 2024-01-21 14:36:00+00:00            18.000000
9            rabbitmq   socket             25 2024-01-21 14:36:00+00:00            11.000000
1            carts-db   socket             25 2024-01-21 14:36:00+00:00            10.000000
14            user-db   socket             25 2024-01-21 14:36:00+00:00             8.000000
2           catalogue   socket             25 2024-01-21 14:36:00+00:00             6.000000
6           orders-db   socket             25 2024-01-21 14:36:00+00:00             5.000000
7             payment   socket             25 2024-01-21 14:36:00+00:00             5.000000
3        catalogue-db   socket             25 2024-01-21 14:36:00+00:00             4.000000
8        queue-master   socket             25 2024-01-21 14:36:00+00:00             3.000000
11         session-db   socket             25 2024-01-21 14:36:00+00:00             3.000000
10  rabbitmq-exporter   socket             25 2024-01-21 14:36:00+00:00             1.000000
5              orders   socket              2 2024-01-21 14:53:00+00:00            16.250000
0               carts   socket              2 2024-01-21 14:37:00+00:00            14.916667
4           front-end   socket              2 2024-01-21 14:49:00+00:00             9.666667
12           shipping   socket              2 2024-01-21 14:55:00+00:00             6.400000, [Timestamp('2024-01-21 14:36:00+0000', tz='UTC'), Timestamp('2024-01-21 14:37:00+0000', tz='UTC'), Timestamp('2024-01-21 14:49:00+0000', tz='UTC'), Timestamp('2024-01-21 14:53:00+0000', tz='UTC'), Timestamp('2024-01-21 14:55:00+0000', tz='UTC')])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse cached metric_df and p95_df (compute p95_df if not present)
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
if 'p95_df' not in globals():
    p95_df = metric_df.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'p95'})

# Ensure types and ts column
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Incident window
start = pd.to_datetime("2024-01-21 14:33:03", utc=True)
end   = pd.to_datetime("2024-01-21 15:03:03", utc=True)

# Filter socket KPI within window
socket_window = metric_df[
    (metric_df['kpi_name'] == 'socket') &
    (metric_df['ts'] >= start) &
    (metric_df['ts'] <= end)
].copy()

# Merge global p95 for socket KPIs
socket_window = socket_window.merge(
    p95_df[p95_df['kpi_name']=='socket'][['cmdb_id','kpi_name','p95']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Determine anomaly where p95 exists and value >= p95
socket_window['is_anomaly'] = False
mask_p95 = socket_window['p95'].notna()
socket_window.loc[mask_p95, 'is_anomaly'] = socket_window.loc[mask_p95, 'value'] >= socket_window.loc[mask_p95, 'p95']

# Group by minute timestamp to count distinct services breaching their p95 and collect examples
# Use the original integer 'timestamp' (unix seconds) as requested
anom_rows = socket_window[socket_window['is_anomaly']].copy()

grouped = (
    anom_rows
    .groupby(['timestamp','ts'], as_index=False)
    .agg(
        socket_anomaly_count = ('cmdb_id', lambda s: s.nunique()),
        examples = ('cmdb_id', lambda s: ','.join(sorted(pd.unique(s).astype(str))[:10]))
    )
)

# Sort by descending count then timestamp, take top 10
result_top10 = grouped.sort_values(['socket_anomaly_count','timestamp'], ascending=[False, True]).head(10)

# Find timestamp(s) with maximum socket_anomaly_count
if not result_top10.empty:
    max_count = int(result_top10['socket_anomaly_count'].max())
    max_rows = result_top10[result_top10['socket_anomaly_count'] == max_count][['timestamp','ts','socket_anomaly_count','examples']].reset_index(drop=True)
else:
    max_count = 0
    max_rows = pd.DataFrame(columns=['timestamp','ts','socket_anomaly_count','examples'])

# Display compact outputs
result_top10, max_count, max_rows

```
Out[4]:
```
Compact summary:

- I counted, for each minute in the incident window, how many distinct services had socket value >= their global P95.

- Top timestamps (unix -> UTC) with highest distinct-service counts:
  1. 1705848900 -> 2024-01-21 14:55:00 UTC — socket_anomaly_count = 13 (examples: carts-db, catalogue, catalogue-db, orders, orders-db, payment, rabbitmq, user, user-db, session-db, …)
  2. 1705847820 -> 2024-01-21 14:37:00 UTC — socket_anomaly_count = 12
  3. 1705848540 -> 2024-01-21 14:49:00 UTC — socket_anomaly_count = 12
  4. 1705848600 -> 2024-01-21 14:50:00 UTC — socket_anomaly_count = 12
  5. 1705848780 -> 2024-01-21 14:53:00 UTC — socket_anomaly_count = 12
  6. 1705848960 -> 2024-01-21 14:56:00 UTC — socket_anomaly_count = 12
  7. 1705849140 -> 2024-01-21 14:59:00 UTC — socket_anomaly_count = 12
  8. 1705847760 -> 2024-01-21 14:36:00 UTC — socket_anomaly_count = 11
  9. 1705847880 -> 2024-01-21 14:38:00 UTC — socket_anomaly_count = 11
 10. 1705847940 -> 2024-01-21 14:39:00 UTC — socket_anomaly_count = 11

- Maximum socket_anomaly_count = 13, occurring at 2024-01-21 14:55:00 UTC.

- Interpretation: many services hit or exceeded their historical P95 socket levels nearly simultaneously across the 14:36–14:59 UTC period, peaking at 14:55 UTC — this indicates a broad, near-system-wide socket-level saturation event.

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp                        ts  socket_anomaly_count                                           examples
19  1705848900 2024-01-21 14:55:00+00:00                    13  carts-db,catalogue,catalogue-db,orders,orders-...
1   1705847820 2024-01-21 14:37:00+00:00                    12  carts,carts-db,catalogue,catalogue-db,orders-d...
13  1705848540 2024-01-21 14:49:00+00:00                    12  carts-db,catalogue,catalogue-db,front-end,orde...
14  1705848600 2024-01-21 14:50:00+00:00                    12  carts-db,catalogue,catalogue-db,front-end,orde...
17  1705848780 2024-01-21 14:53:00+00:00                    12  carts-db,catalogue,catalogue-db,orders,orders-...
20  1705848960 2024-01-21 14:56:00+00:00                    12  carts-db,catalogue,catalogue-db,orders-db,paym...
23  1705849140 2024-01-21 14:59:00+00:00                    12  carts,carts-db,catalogue,catalogue-db,orders-d...
0   1705847760 2024-01-21 14:36:00+00:00                    11  carts-db,catalogue,catalogue-db,orders-db,paym...
2   1705847880 2024-01-21 14:38:00+00:00                    11  carts-db,catalogue,catalogue-db,orders-db,paym...
3   1705847940 2024-01-21 14:39:00+00:00                    11  carts-db,catalogue,catalogue-db,orders-db,paym..., 13,     timestamp                        ts  socket_anomaly_count                                           examples
0  1705848900 2024-01-21 14:55:00+00:00                    13  carts-db,catalogue,catalogue-db,orders,orders-...)```
```